# Exercice 2 — Placement de k jetons sur une grille n×n

Chaque ligne et chaque colonne doit contenir un nombre **pair** de jetons. Données de l'exemple : `n = 5`, `k = 10`.

In [1]:
import random
from math import comb

n = 5
k = 10
TAILLE_GRILLE = n * n

## 1) Calculer la taille de l'espace de recherche
On choisit k cases parmi les n² cases, sans tenir compte de l'ordre.

In [2]:
taille_espace_recherche = comb(n * n, k)
print("Taille de l'espace de recherche :", taille_espace_recherche)

Taille de l'espace de recherche : 3268760


## 2.a) Proposer un codage des solutions
Un individu est une liste de `k` positions distinctes parmi les `n*n` cases.

In [3]:
def creer_individu():
    return random.sample(range(TAILLE_GRILLE), k)

def creer_population(taille_population):
    return [creer_individu() for _ in range(taille_population)]

def convertir_en_grille(individu):
    grille = [[0 for _ in range(n)] for _ in range(n)]
    for position in individu:
        ligne = position // n
        colonne = position % n
        grille[ligne][colonne] = 1
    return grille

## 2.b) Proposer une fonction fitness
On compte les lignes et les colonnes qui ont un nombre **impair** de jetons. La fitness est une pénalité à **minimiser**. Une solution faisable a une fitness égale à 0.

In [4]:
def fonction_fitness(individu):
    grille = convertir_en_grille(individu)

    nb_lignes_impaires = 0
    for ligne in grille:
        if sum(ligne) % 2 != 0:
            nb_lignes_impaires += 1

    nb_colonnes_impaires = 0
    for j in range(n):
        total = sum(grille[i][j] for i in range(n))
        if total % 2 != 0:
            nb_colonnes_impaires += 1

    return nb_lignes_impaires + nb_colonnes_impaires

## 2.c) Opérateurs génétiques et code complet
- **Sélection par tournoi :** on garde le candidat ayant la plus petite pénalité.
- **Croisement :** on conserve des positions des deux parents, puis on complète jusqu'à k positions distinctes.
- **Mutation :** on déplace un jeton vers une case vide. Le nombre total de jetons reste égal à k.
- **Élitisme :** on conserve la meilleure solution.

In [5]:
def selection_tournoi(population, taille_tournoi=3):
    candidats = random.sample(population, taille_tournoi)
    return min(candidats, key=fonction_fitness)

def croisement(parent1, parent2):
    positions_communes = list(set(parent1) & set(parent2))
    autres_positions = list((set(parent1) | set(parent2)) - set(positions_communes))
    random.shuffle(autres_positions)

    enfant = positions_communes + autres_positions[:k - len(positions_communes)]

    while len(enfant) < k:
        position = random.randrange(TAILLE_GRILLE)
        if position not in enfant:
            enfant.append(position)

    return enfant

def mutation(individu, pm=0.3):
    nouvel_individu = individu[:]

    if random.random() < pm:
        positions_occupees = set(nouvel_individu)
        positions_libres = [p for p in range(TAILLE_GRILLE)
                            if p not in positions_occupees]
        indice = random.randrange(k)
        nouvel_individu[indice] = random.choice(positions_libres)

    return nouvel_individu

In [6]:
def algorithme_genetique(taille_population=100, max_generations=500,
                         pc=0.8, pm=0.3):
    population = creer_population(taille_population)
    meilleure_solution = min(population, key=fonction_fitness)

    for generation in range(max_generations):
        nouvelle_population = [meilleure_solution[:]]  # élitisme

        while len(nouvelle_population) < taille_population:
            parent1 = selection_tournoi(population)
            parent2 = selection_tournoi(population)

            if random.random() < pc:
                enfant = croisement(parent1, parent2)
            else:
                enfant = parent1[:]

            enfant = mutation(enfant, pm)
            nouvelle_population.append(enfant)

        population = nouvelle_population
        candidat = min(population, key=fonction_fitness)

        if fonction_fitness(candidat) < fonction_fitness(meilleure_solution):
            meilleure_solution = candidat[:]

        if fonction_fitness(meilleure_solution) == 0:
            print('Solution faisable à la génération', generation)
            break

    return meilleure_solution

## Exécution et affichage de la solution

In [11]:
def afficher_grille(individu):
    grille = convertir_en_grille(individu)
    for ligne in grille:
        print(' '.join('●' if case == 1 else '·' for case in ligne))

solution = algorithme_genetique()
print('Fitness (0 = solution faisable) :', fonction_fitness(solution))
print('Nombre de jetons :', len(solution))
afficher_grille(solution)

Solution faisable à la génération 1
Fitness (0 = solution faisable) : 0
Nombre de jetons : 10
· · ● · ●
· · ● · ●
● · · ● ·
● · · · ●
· · · ● ●
